# Day 1 Block 3 — frozen CLIP extraction (Kaggle T4)

**Before Run All:**
1. Settings → Accelerator → **GPU T4**.
2. Add input → `aigc-payload` **and** `aigc-code`.
3. Add-ons → Secrets → `HF_TOKEN`.

Outputs land in `/kaggle/working/cache` (+ `.meta.json` per cache with img/s). After the run: **Save Version** to persist caches as an output dataset.

In [ ]:
!pip install -q open-clip-torch timm ImageHash
import torch
print(torch.__version__, '|', torch.cuda.get_device_name(0), '| cuda:', torch.cuda.is_available())

In [ ]:
# HF Hub auth: needs HF_TOKEN attached under Add-ons → Secrets.
# Exports it to the environment so huggingface_hub/open_clip pick it up.
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ.setdefault('HF_TOKEN', UserSecretsClient().get_secret('HF_TOKEN'))
    print('HF_TOKEN:', 'set' if os.environ.get('HF_TOKEN') else 'EMPTY')
except Exception as e:
    print('no kaggle secret available:', e)

In [ ]:
# Stage code (clone) + data (symlink: no 2.7 GB copy; inputs are read-only).
!rm -rf /kaggle/working/repo && mkdir -p /kaggle/working/cache
!git clone -q -b feat/features https://github.com/yuletan/AI-Image-Detection.git /kaggle/working/repo
import glob, os, shutil
man = glob.glob('/kaggle/input/**/manifest.csv', recursive=True)
assert len(man) == 1, man
MANIFEST = man[0]
link = '/kaggle/working/repo/data'
if os.path.islink(link) or os.path.isfile(link):
    os.remove(link)
elif os.path.isdir(link):
    shutil.rmtree(link)  # clone recreates data/ from data/README.md
os.symlink(os.path.join(os.path.dirname(MANIFEST), 'data'), link)
print('MANIFEST =', MANIFEST)
!ls /kaggle/working/repo/data/raw | head

In [ ]:
# Smoke test: 512 images. Track 3 expects ~80-150 img/s on T4.
# If slow, raise --batch-size (128/256) and re-run before the full loop.
!cd /kaggle/working/repo && PYTHONPATH=src python -m aigc_detect.features.extract --split test --transform clean --limit 512 --manifest {MANIFEST} --cache-dir /kaggle/working/cache --batch-size 64 --workers 2
!cat /kaggle/working/cache/test_clean_None.npy.meta.json

In [ ]:
# Full Block 3 loop: clean train/val/test/heldout + every table param on test
# + both chains on test. Skips caches that already exist (timeout resume).
# Reuses MANIFEST from the staging cell above — run cells top to bottom.
import os, subprocess, sys, time, yaml
from pathlib import Path
sys.path.insert(0, '/kaggle/working/repo/src')
from aigc_detect.features import cache_path

REPO = '/kaggle/working/repo'
CACHE = '/kaggle/working/cache'
BS, WORKERS = '128', '4'  # set from the smoke test above

cfg = yaml.safe_load(open(f'{REPO}/configs/transforms.yaml'))
jobs = [{'split': s, 'transform': 'clean', 'param': None, 'chain': None}
        for s in ['train', 'val', 'test', 'heldout']]
jobs += [{'split': 'test', 'transform': t, 'param': p, 'chain': None}
         for t, ps in cfg['transforms'].items() if t != 'clean' for p in ps]
jobs += [{'split': 'test', 'transform': c, 'param': 'chain', 'chain': c}
         for c in cfg.get('chains', {})]
print(f'{len(jobs)} jobs')

t0 = time.perf_counter()
for j in jobs:
    target = f"{CACHE}/" + cache_path(Path(CACHE), j['split'], j['transform'], j['param']).name
    if os.path.exists(target):
        print(f'skip (exists): {os.path.basename(target)}', flush=True)
        continue
    cmd = [sys.executable, '-m', 'aigc_detect.features.extract',
           '--split', j['split'], '--manifest', MANIFEST, '--cache-dir', CACHE,
           '--batch-size', BS, '--workers', WORKERS, '--skip-broken']
    cmd += ['--chain', j['chain']] if j['chain'] else ['--transform', j['transform']]
    if not j['chain'] and j['param'] is not None:
        cmd += ['--param', str(j['param'])]
    print('$', ' '.join(cmd[4:8]), flush=True)
    r = subprocess.run(cmd, cwd=REPO, env={**os.environ, 'PYTHONPATH': 'src'})
    if r.returncode != 0:
        raise RuntimeError(f"FAILED: {target} — fix and re-run (caches resume)")
print(f'ALL DONE in {(time.perf_counter() - t0) / 60:.1f} min')

## After the loop
1. `!ls -la /kaggle/working/cache | head` — expect ~21 `.npy` + index/meta sidecars.
2. **Save Version** (top right) so caches persist as a versioned output.
3. Paste the `imgs_per_sec` values back — Day-1 Block 4 (linear probe) is CPU work and runs locally.